# Phase 3 — Incremental Processing, CDC and Idempotency (PySpark)

We will extend the same e-commerce project without modifying Phases 1 or 2.

**Pipeline:** existing Silver Parquet snapshot + simulated change events → deduplicate to latest event → validate upserts → apply inserts, updates and deletes → write new Silver Parquet snapshot → refresh Gold summaries.

**What you'll learn:** incremental processing, Change Data Capture (CDC) concepts, upserts, delete tombstones, event ordering, exactly-once *effects* through idempotent replay, snapshot versioning and validation.

This uses **simulated CDC from Python**, not an actual Debezium/Kafka stream or Delta Lake MERGE. Outputs in Colab are temporary until you copy them to Google Drive.

## Step 1 — Install PySpark and retrieve your GitHub project

We clone the existing GitHub project and import the Phase 2 and Phase 3 modules. Use **Runtime → Run all** or execute each cell to understand the steps.

In [ ]:
%pip -q install "pyspark>=3.5,<4"

In [ ]:
from pathlib import Path
import subprocess, sys
PROJECT = Path("/content/ecommerce-data-engineering-pipeline")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/mosesrathan19/ecommerce-data-engineering-pipeline.git",
                    str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
sys.path.insert(0, str(PROJECT))
from src.spark_pipeline import spark_session
from src.incremental_pipeline import generate_cdc_events, latest_events, apply_cdc, same_rows, run_demo
from pyspark.sql import functions as F
spark = spark_session()
print("Spark version:", spark.version)
print("Project:", PROJECT)

## Step 2 — Initialize the baseline Silver Parquet

In Phase 2 we had **99,375** accepted orders. The Phase 3 runner will build its own baseline from the original CSV data if necessary, so this notebook also works in a brand-new Colab session. This is the **initial one-time full load**, not the daily incremental workload.

In [ ]:
OUTPUT = PROJECT / "outputs" / "incremental"
report, snapshot = run_demo(spark, PROJECT, OUTPUT)
print("Phase 3 batch report:", report)
snapshot.select("order_id", "quantity", "status", "recognized_revenue").show(5, truncate=False)

## Step 3 — Examine our simulated CDC events

A CDC event describes a change to an existing record or a new record.

- **I (insert):** A new order ID.
- **U (update):** A revised version of an existing order.
- **D (delete):** A tombstone; the order should disappear from the current-state snapshot.

The demo produces 500 inserts, 40 updates, 10 deletes and 10 *older* duplicate updates (560 input events). Those last 10 help us test event-time ordering.

In [ ]:
base = spark.read.parquet(str(OUTPUT / "baseline" / "orders"))
events = generate_cdc_events(spark, base)
print("Total events:", events.count())
events.groupBy("op").count().orderBy("op").show()
events.orderBy("order_id", "event_ts").show(12, truncate=False)

## Step 4 — Pick the latest event for each order

Older events must not overwrite a newer change. Here we use `row_number` with a Spark window ordered by event timestamp, highest first.

For one order with changes at 11:00 AM and 12:00 PM, we keep the 12:00 PM event. We also reject ambiguous events that have the same ID and timestamp.

In [ ]:
chosen = latest_events(events)
print("Incoming raw events:", events.count())
print("Effective latest events:", chosen.count())
chosen.groupBy("op").count().orderBy("op").show()

## Step 5 — Apply upserts and deletes

**Upsert = update if the order exists, otherwise insert.**

We validate I/U rows using Phase 2's existing Silver cleaning logic. Then a Spark **left-anti join** keeps only unchanged old rows. Finally, `unionByName` adds the valid new/current rows.

For delete tombstones, the old row is removed but no replacement is inserted.

In [ ]:
customers = spark.read.option("header", True).csv(str(PROJECT / "data/raw/customers.csv"))
products = spark.read.option("header", True).csv(str(PROJECT / "data/raw/products.csv"))
merged, effective = apply_cdc(base, events, customers, products)
print("Baseline rows:", base.count())
print("After CDC:", merged.count())
print("Distinct order IDs:", merged.select("order_id").distinct().count())
assert merged.count() == 99865

## Step 6 — Prove idempotency

**Idempotent means applying the same batch again has the same final result.**

We apply the exact same CDC events to the already merged data. The data should be identical, including every value, not just the count.

In [ ]:
replayed, _ = apply_cdc(merged, events, customers, products)
assert same_rows(merged, replayed), "Replay changed the dataset!"
print("Idempotency test passed.")
print("First application:", merged.count(), "rows")
print("Replay:", replayed.count(), "rows")

## Step 7 — Inspect the persisted Parquet snapshot and Gold tables

The runner wrote a **new version** to `outputs/incremental/snapshots/2026-11-20-demo/orders/`. It did **not** overwrite the baseline. It also wrote a JSON manifest recording the batch and three Gold Parquet aggregates.

In this educational example, a completed-batch manifest prevents a second full run of the same batch ID. This is **not** a transactional multi-writer design; production jobs need stronger storage transactions or table formats such as Delta Lake or Iceberg.

In [ ]:
from src.spark_pipeline import gold_tables
batch_path = OUTPUT / "snapshots" / "2026-11-20-demo"
saved = spark.read.parquet(str(batch_path / "orders"))
print("Saved snapshot count:", saved.count())
assert same_rows(saved, merged)
for name in ["monthly_revenue", "category_revenue", "state_revenue"]:
    frame = spark.read.parquet(str(batch_path / "gold" / name))
    print(name)
    frame.show(5, truncate=False)
print("Manifest:", (batch_path / "manifest.json").read_text())

## Step 8 — Rerun the completed batch safely

We call `run_demo` again. It sees the completed batch manifest and returns the same saved Parquet snapshot, without appending duplicate orders or overwriting the data.

In [ ]:
again_report, again_snapshot = run_demo(spark, PROJECT, OUTPUT)
assert same_rows(saved, again_snapshot)
assert again_report == report
print("Re-running the batch produced identical data:", again_snapshot.count(), "rows")
print("Phase 3 complete ✅")

## Step 9 — Spark SQL on the current snapshot

In a real system, downstream reporting should read the **current committed snapshot**, not raw CDC events. Query the saved Silver Parquet using Spark SQL.

In [ ]:
saved.createOrReplaceTempView("orders_current")
spark.sql("""
SELECT category,
       COUNT(*) AS order_count,
       ROUND(SUM(recognized_revenue), 2) AS revenue
FROM orders_current
GROUP BY category
ORDER BY revenue DESC
""").show(truncate=False)

## Next steps / Production lessons

- **CDC source:** Replace the simulated events with Debezium/Kafka, database log CDC, or a managed ingestion connector.
- **Checkpoints:** Persist source offsets, successful batch IDs and consumer state.
- **Transactional tables:** Delta Lake or Iceberg supports safer snapshot commits and MERGE operations than plain folders of Parquet files.
- **Incremental Gold:** Here Gold tables are rebuilt from the merged Silver snapshot; later optimize only affected partitions.
- **Late-arriving events:** Persist the latest event version/timestamp per order so even events from a *later batch* cannot override a newer event.
- **Durability:** Colab local files are ephemeral; copy output to Drive or cloud object storage.